# Aplicação Prática — Relatório de Conformidade de EPI

Transforma as detecções de violação de EPI do vídeo em um relatório de auditoria automática de segurança (eventos com timestamp, duração e confiança). Requer o vídeo final e os pesos do detector (`05_inference_video.ipynb` e `02_detection.ipynb`).

## Setup

In [ ]:
!pip install -q ultralytics opencv-python pandas pyarrow matplotlib pillow kaggle

from pathlib import Path

# Armazenamento persistente compartilhado entre os notebooks: monta o Google
# Drive e usa uma pasta fixa. Troque o caminho se preferir outra estrutura.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/vc-seguranca-trabalho')
except ImportError:
    # Execucao fora do Colab (teste local) - usa uma pasta local.
    PROJECT_DIR = Path('./vc-seguranca-trabalho').resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
ROOT = PROJECT_DIR
print("Diretorio do projeto:", ROOT)


## 14. Detectar violações e gerar o relatório

Roda o detector quadro a quadro, agrupa detecções consecutivas de `NO-Hardhat`, `NO-Mask` e `NO-Safety Vest` em eventos, e salva os dados brutos em Parquet.

In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO

VIDEO_INPUT = ROOT / "video" / "input" / "video_final_canteiro_obra.mp4"
DETECTION_WEIGHTS = ROOT / "models" / "detection" / "css_yolov8n_baseline" / "weights" / "best.pt"
REPORTS_DIR = ROOT / "reports"

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]
VIOLATION_CLASSES = {"NO-Hardhat", "NO-Mask", "NO-Safety Vest"}
CONF_THRESHOLD = 0.25
MAX_GAP_SECONDS = 0.5


def get_video_fps(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    return fps


def collect_frame_detections(fps):
    model = YOLO(str(DETECTION_WEIGHTS))
    detections = []
    frame_idx = 0
    for result in model.predict(source=str(VIDEO_INPUT), conf=CONF_THRESHOLD, stream=True, verbose=False):
        timestamp = frame_idx / fps
        for cls_id, conf in zip(result.boxes.cls.tolist(), result.boxes.conf.tolist()):
            class_name = CLASS_NAMES[int(cls_id)]
            if class_name in VIOLATION_CLASSES:
                detections.append((timestamp, class_name, conf))
        frame_idx += 1
    return detections


def group_into_events(detections):
    by_class = {}
    for ts, cls, conf in detections:
        by_class.setdefault(cls, []).append((ts, conf))

    events = []
    for cls, items in by_class.items():
        items.sort(key=lambda x: x[0])
        current_start = items[0][0]
        current_end = items[0][0]
        current_confs = [items[0][1]]

        for ts, conf in items[1:]:
            if ts - current_end <= MAX_GAP_SECONDS:
                current_end = ts
                current_confs.append(conf)
            else:
                events.append((cls, current_start, current_end, sum(current_confs) / len(current_confs)))
                current_start = ts
                current_end = ts
                current_confs = [conf]

        events.append((cls, current_start, current_end, sum(current_confs) / len(current_confs)))

    events.sort(key=lambda e: e[1])
    return events


def write_parquet(events, out_path):
    df = pd.DataFrame([
        {
            "classe": cls, "inicio_s": round(start, 2), "fim_s": round(end, 2),
            "duracao_s": round(end - start, 2), "confianca_media": round(conf, 3),
        }
        for cls, start, end, conf in events
    ])
    df.to_parquet(out_path, engine="pyarrow", index=False)
    return df


fps = get_video_fps(VIDEO_INPUT)
print(f"FPS do vídeo: {fps}")

print("Rodando detector quadro a quadro (stream=True)...")
detections = collect_frame_detections(fps)
print(f"Detecções de violação capturadas: {len(detections)}")

events = group_into_events(detections)
print(f"Eventos agregados: {len(events)}")

cap = cv2.VideoCapture(str(VIDEO_INPUT))
video_duration = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
cap.release()

parquet_path = REPORTS_DIR / "violacoes-epi.parquet"
df = write_parquet(events, parquet_path)
print(f"Parquet salvo: {parquet_path}")
df


## Resumo por classe

In [ ]:
# Resumo por classe (a partir do DataFrame gerado acima)
resumo = df.groupby("classe").agg(
    eventos=("classe", "count"),
    tempo_total_s=("duracao_s", "sum"),
).reset_index()
resumo["pct_do_video"] = (resumo["tempo_total_s"] / video_duration * 100).round(1)
resumo


## Aplicação prática

Este relatório é uma auditoria automática de segurança: em vez de assistir ao vídeo inteiro, um responsável de segurança do trabalho consulta diretamente os eventos acima para saber exatamente quando cada EPI esteve ausente.

Ver `docs/relatorio-tecnico.md` seção 7 para os resultados completos obtidos no desenvolvimento original (ex. NO-Safety Vest presente em 62.8% do vídeo analisado).